In [14]:
import pandas as pd
import numpy as np

In [15]:
# get needed sheets only
stores = list(pd.read_excel("تصنيف العملاء_2025-2024(1).xlsx",sheet_name=None).keys())[:-2]

# load all dataframes in a dictionary
df_dict = pd.read_excel("تصنيف العملاء_2025-2024(1).xlsx",sheet_name=stores)

# this line is just renaming the branches for output on next cell
df_dict = {f"branch_{j}" : k for j,k in enumerate(df_dict.values(),start=1)}

In [ ]:
# raw data example
df_dict["branch_1"]

,Unnamed: 0,2024,Unnamed: 2,Unnamed: 3,2025,Unnamed: 5,Unnamed: 6
0,month,عدد الفواتير,محقق شهريا,متوسط الفاتورة,عدد الفواتير,محقق شهريا,متوسط الفاتورة
1,Jan,101,127651,1263.871287,95,120568,1269.136842
2,Feb,77,139703,1814.324675,93,160850,1729.569892
3,Mar,82,130673,1593.573171,78,93050,1192.948718
4,Apr,80,116850,1460.625,81,162250,2003.08642
5,May,88,117828,1338.954545,64,97832,1528.625
6,Jun,101,135266,1339.267327,61,66330,1087.377049
7,Jul,74,118312.5,1598.817568,110,149150,1355.909091
8,Aug,64,110191.5,1721.742188,111,151850,1368.018018
9,Sep,91,94200,1035.164835,88,105300,1196.590909


In [ ]:
# main loop to clean each sheet
for i in df_dict:
    # renaming columns and dropping extra headers
    df_dict[i].columns = ["month","2024_عدد الفواتير","drop1","2024_محقق شهريا","2025_عدد الفواتير","2025_محقق شهريا","drop2"]
    df_dict[i] = df_dict[i].drop(df_dict[i].index[0]).reset_index(drop=True)
    
    # add a column for branch name and get only needed sheets
    df_dict[i]["branch_name"] = i.strip()
    df_dict[i] = df_dict[i][['branch_name','month', '2024_عدد الفواتير', '2024_محقق شهريا','2025_عدد الفواتير', '2025_محقق شهريا']]
    
    # extract first twelve rows only (months) and convert revenue columns to float
    df_dict[i] = df_dict[i].head(12)
    df_dict[i][list(df_dict[i])[2:]] = df_dict[i][list(df_dict[i])[2:]].astype("float").round(2)

# combine all branches in one single table
branches_monthly_recipts_earnings = pd.concat(df_dict.values(), ignore_index=True)

# output
# branches_monthly_recipts_earnings.to_excel("out/branches_monthly_recipts_earnings.xlsx",sheet_name="branch_monthly income",index=False)

In [ ]:
# table sample after cleaning
branches_monthly_recipts_earnings

,branch_name,month,2024_عدد الفواتير,2024_محقق شهريا,2025_عدد الفواتير,2025_محقق شهريا
0,branch_1,Jan,101.0,1263.87,95.0,120568.0
1,branch_1,Feb,77.0,1814.32,93.0,160850.0
2,branch_1,Mar,82.0,1593.57,78.0,93050.0
3,branch_1,Apr,80.0,1460.62,81.0,162250.0
4,branch_1,May,88.0,1338.95,64.0,97832.0
...,...,...,...,...,...,...
211,branch_18,Aug,33.0,1604.55,20.0,56600.0
212,branch_18,Sep,20.0,2115.00,24.0,39750.0
213,branch_18,Oct,21.0,2673.52,17.0,23535.0
214,branch_18,nov,18.0,1476.39,15.0,13450.0


In [ ]:
# cleaning monthly sales sheet, same as before with a few tweaks

stores2 = list(pd.read_excel("تقرير حجم نمو 2025(1).xlsx",sheet_name=None).keys())[:-2]
df_dict2 = pd.read_excel("تقرير حجم نمو 2025(1).xlsx",sheet_name=stores2)

for i in df_dict2:
    df_dict2[i] = df_dict2[i].drop(index=[0,1,2,3,4]).dropna(how="all").replace({"New In 2024":0,"Closed":0,"New Branch":0,np.nan:0}).reset_index(drop=True)
    df_dict2[i].columns = ["branch_code","branch_name","sales_2024","sales_2025","drop1","drop2"]
    df_dict2[i].drop(columns=["drop1","drop2"],inplace=True)
    df_dict2[i]["month"] = i.strip()
    df_dict2[i] = df_dict2[i][["branch_code","branch_name","month","sales_2024","sales_2025"]]

branch_monthly_sales_per_year = pd.concat(df_dict2.values(), ignore_index=True)

# branch_monthly_sales_per_year.to_excel("out/branch_monthly_sales_per_year.xlsx",sheet_name="branch_monthly income",index=False)

In [ ]:
branch_monthly_sales_per_year

In [ ]:
# cleaning recipt status sheet, same

def name_update(x):
    if "businees_name" in x:
        return x
    else:
        return "businees_name "+x

stores3_1 = list(pd.read_excel("حجم نمو العملا 20250.xlsx",sheet_name=None).keys())[:-1]
stores3_2 = list(pd.read_excel("حجم نمو العملاء 2024 2025.xlsx",sheet_name=None).keys())[:-1]
df_dict3 = pd.read_excel("حجم نمو العملا 20250.xlsx",sheet_name=stores3_1) | pd.read_excel("حجم نمو العملاء 2024 2025.xlsx",sheet_name=stores3_2)

for i in df_dict3:
    df_dict3[i].columns = ["month","منتهي_2024","مرفوض_2024","منتهي_2025","مرفوض_2025","drop1","drop2","drop3"]
    df_dict3[i].drop(index=0,axis=0,inplace=True)
    df_dict3[i].drop(columns=["drop1","drop2","drop3"],inplace=True)
    df_dict3[i] = df_dict3[i].head(12).reset_index(drop=True)
    df_dict3[i].replace({np.nan:0},inplace=True)
    df_dict3[i]["branch_name"] = i.strip()
    df_dict3[i] = df_dict3[i][["branch_name","month","منتهي_2024","مرفوض_2024","منتهي_2025","مرفوض_2025"]]

branch_monthly_recipt_status = pd.concat(df_dict3.values(), ignore_index=True)

branch_monthly_recipt_status["branch_name"] = branch_monthly_recipt_status["branch_name"].apply(name_update)

# branch_monthly_recipt_status.to_excel("out/branch_monthly_recipt_status.xlsx",index=False)

In [ ]:
branch_monthly_recipt_status